# Week 31, Day 3: Adaptation and firing patterns: the AdEx model

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w031/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Classifying patterns automatically

`adex(I, a, b, tw, Vr)` returns spike times for a 600 ms simulation with the current switched on at 50 ms. Write `adaptation_index(isi)` and `classify(spikes)`: phasic if fewer than 2 spikes or the last spike comes more than 200 ms before the end; bursting if the longest interval is more than 8 times the shortest; adapting if the adaptation index exceeds 0.02; tonic otherwise.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

def adex(I, a, b, tw, Vr, T=600.0, dt=0.05, C=200.0, gL=10.0, EL=-70.0, VT=-50.0, DT=2.0, trace=False):
    V, w, spikes, Vs = EL, 0.0, [], []
    for k in range(int(T / dt)):
        t = k * dt
        Iext = I if t >= 50 else 0.0
        dV = (-gL * (V - EL) + gL * DT * math.exp((V - VT) / DT) - w + Iext) / C
        dw = (a * (V - EL) - w) / tw
        V += dt * dV; w += dt * dw
        if V > 0:
            V = Vr; w += b; spikes.append(t)
        if trace:
            Vs.append(min(V, 0.0))
    return (np.array(spikes), np.array(Vs)) if trace else np.array(spikes)

def adaptation_index(isi):
    return ...

def classify(spikes, T=600.0):
    ...

sets = {"tonic": (0, 5, 30, -58, 500), "adapting": (0, 60, 200, -58, 500),
        "bursting": (0, 60, 100, -45, 300), "phasic": (30, 10, 200, -58, 350)}   # a (nS), b (pA), tau_w (ms), V_r (mV), I (pA)
labels = {}
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(6, 6))
for ax, (name, (a, b, tw, Vr, I)) in zip(axes, sets.items()):
    s, v = adex(I, a, b, tw, Vr, trace=True)
    labels[name] = classify(s)
    ax.plot(np.arange(len(v)) * 0.05, v); ax.set_ylabel(name)
    print(f"{name:9s}: {len(s):2d} spikes, first intervals {np.round(np.diff(s)[:6], 1)}, classified as {labels[name]}")
axes[-1].set_xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_index():
    "adaptation_index averages (ISI[k+1] - ISI[k]) / (ISI[k+1] + ISI[k])"
    assert abs(adaptation_index(np.array([10.0, 20.0, 20.0])) - (1 / 3 + 0) / 2) < 1e-12

def test_classes():
    "the four parameter sets are classified as tonic, adapting, bursting and phasic"
    assert labels == {"tonic": "tonic", "adapting": "adapting", "bursting": "bursting", "phasic": "phasic"}

test_index()
test_classes()
print("All checks passed")

### Exercise: Sweeping the reset

Using the bursting set's a, b and τ_w at 300 pA, vary the reset from -60 to -44 mV. Group spikes into bursts (a new burst starts after an interval longer than 20 ms) and store the median number of spikes per burst for each reset in `spikes_per_burst`.

In [ ]:
resets = [-60, -55, -50, -48, -46, -45, -44]
spikes_per_burst = {}
for Vr in resets:
    s = adex(300, 0, 60, 100, Vr, T=1200)
    isi = np.diff(s)
    starts = ...                                # indices where a new burst begins
    sizes = np.diff(np.append(starts, len(s)))
    spikes_per_burst[Vr] = int(np.median(sizes))
    print(f"V_r = {Vr} mV: {len(s)} spikes, median {spikes_per_burst[Vr]} per burst")

In [ ]:
import numpy as np
def test_sweep():
    "spikes_per_burst has an entry for each reset voltage"
    assert set(spikes_per_burst) == set(resets)

def test_trend():
    "a low reset gives one spike per group and a high reset several"
    assert spikes_per_burst[-60] == 1 and spikes_per_burst[-45] >= 3

test_sweep()
test_trend()
print("All checks passed")